# 06 — Comparative Analysis
// Pipeline A vs Pipeline B — Statistical comparison

**Objective**: Scientifically compare Pipeline A (raw) vs Pipeline B (DSP) performance.

Covers:
- Results table (all models, both pipelines)
- Paired t-test and Wilcoxon signed-rank test
- Effect size (Cohen's d)
- Confusion matrices comparison
- Per-class accuracy analysis
- Box plots of fold-wise accuracy

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pickle
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import config
from src.evaluation import (
    aggregate_fold_results, compare_pipelines, generate_results_table
)

plt.style.use('dark_background')
%matplotlib inline

## 1. Load Results

In [ ]:
with open(os.path.join(config.RESULTS_DIR, 'pipeline_a_results.pkl'), 'rb') as f:
    pipeline_a = pickle.load(f)

with open(os.path.join(config.RESULTS_DIR, 'pipeline_b_results.pkl'), 'rb') as f:
    pipeline_b = pickle.load(f)

print("Results loaded.")

## 2. Results Comparison Table

In [ ]:
all_results = {
    ('SVM', 'A (Raw)'): pipeline_a['svm_agg'],
    ('SVM', 'B (DSP)'): pipeline_b['svm_agg'],
    ('Random Forest', 'A (Raw)'): pipeline_a['rf_agg'],
    ('Random Forest', 'B (DSP)'): pipeline_b['rf_agg'],
    ('CNN-2D', 'A (Raw)'): pipeline_a['cnn_agg'],
    ('CNN-2D', 'B (DSP)'): pipeline_b['cnn_agg'],
}

table = generate_results_table(all_results)
df_results = pd.DataFrame(table)
print(df_results.to_string(index=False))

# Save as CSV
df_results.to_csv(os.path.join(config.TABLES_DIR, 'results_comparison.csv'), index=False)

## 3. Statistical Significance Tests

In [ ]:
for model_name, key in [('SVM', 'svm'), ('Random Forest', 'rf'), ('CNN-2D', 'cnn')]:
    print(f"\n=== {model_name} ===")
    comparison = compare_pipelines(pipeline_a[key], pipeline_b[key], metric='accuracy')
    print(f"  Pipeline A mean: {comparison['pipeline_a_mean']*100:.2f}%")
    print(f"  Pipeline B mean: {comparison['pipeline_b_mean']*100:.2f}%")
    print(f"  Difference:      {comparison['difference']*100:+.2f}%")
    print(f"  Paired t-test:   t={comparison['paired_t_test']['t_stat']:.3f}, "
          f"p={comparison['paired_t_test']['p_value']:.4f}")
    print(f"  Wilcoxon test:   p={comparison['wilcoxon_test']['p_value']:.4f}")
    print(f"  Cohen's d:       {comparison['cohens_d']:.3f}")
    print(f"  Significant (α=0.05): {'YES' if comparison['significant_005'] else 'NO'}")

## 4. Box Plot — Fold-wise Accuracy

In [ ]:
# Prepare data for box plot
box_data = []
for model, key in [('SVM', 'svm'), ('RF', 'rf'), ('CNN', 'cnn')]:
    for fold_m in pipeline_a[key]:
        box_data.append({'Model': model, 'Pipeline': 'A (Raw)', 'Accuracy': fold_m['accuracy']})
    for fold_m in pipeline_b[key]:
        box_data.append({'Model': model, 'Pipeline': 'B (DSP)', 'Accuracy': fold_m['accuracy']})

df_box = pd.DataFrame(box_data)

fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor('#0a0a0a')
ax.set_facecolor('#111111')

sns.boxplot(data=df_box, x='Model', y='Accuracy', hue='Pipeline',
            palette={'A (Raw)': '#888888', 'B (DSP)': '#00ff41'}, ax=ax)
ax.set_title('Pipeline A vs B — Fold-wise Accuracy', color='#00ffff')
ax.set_ylabel('Accuracy', color='#ffffff')
ax.legend(title='Pipeline')
ax.grid(True, alpha=0.2, axis='y')

plt.tight_layout()
plt.savefig(f'{config.FIGURES_DIR}/fig_fold_accuracy_boxplot.png', dpi=150,
            bbox_inches='tight', facecolor='#0a0a0a')
plt.show()

## 5. Accuracy Comparison Bar Chart

In [ ]:
from src.visualization import plot_accuracy_comparison

fig = plot_accuracy_comparison(table, save_name='fig_accuracy_comparison_barplot.png')
plt.show()

## 6. Summary & Key Findings

In [ ]:
print("="*60)
print("KEY FINDINGS")
print("="*60)
print()
print("1. Does DSP preprocessing improve classification performance?")
print("   → Analyze the statistical test results above.")
print()
print("2. Which model benefits most from DSP?")
print("   → Compare difference values across models.")
print()
print("3. Is the improvement statistically significant?")
print("   → Check p-values (α=0.05 threshold).")
print()
print("4. Effect size interpretation:")
print("   Small: |d| < 0.2, Medium: 0.2-0.8, Large: |d| > 0.8")